# Ground Truth Data Generation

Generate question-document pairs for retrieval evaluation.

For each document chunk, we ask gpt-4o-mini to generate 5 questions
that the chunk could answer. This creates our gold standard dataset.

Output: `../data/ground-truth-retrieval.csv`

In [ ]:
import os
import json
import hashlib
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

client = OpenAI()

## 1. Load documents from data/documents/

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def load_documents(docs_path: str = "../data/documents") -> list:
    documents = []
    docs_dir = Path(docs_path)
    for file_path in docs_dir.rglob("*.md"):
        if file_path.name == "README.md":
            continue
        source = file_path.parent.name
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        chunks = chunk_text(content)
        for i, chunk in enumerate(chunks):
            if len(chunk.strip()) < 50:
                continue
            doc_id = hashlib.md5(f"{file_path.name}_{i}".encode()).hexdigest()[:12]
            documents.append({
                "id": doc_id,
                "source": source,
                "filename": file_path.name,
                "chunk_index": i,
                "text": chunk,
            })
    print(f"Loaded {len(documents)} chunks")
    return documents


documents = load_documents()
documents[:2]

## 2. Generate questions with gpt-4o-mini

In [ ]:
PROMPT_TEMPLATE = """
You emulate a DataOps practitioner using a knowledge assistant about dbt, Airflow, and Great Expectations.
Formulate 5 questions this user might ask based on the provided documentation chunk.
Make the questions specific and answerable from the chunk.
Use as few words as possible from the chunk directly.

Source tool: {source}
File: {filename}

Documentation chunk:
{text}

Provide output as parsable JSON without code blocks:
{{"questions": ["question1", "question2", "question3", "question4", "question5"]}}
""".strip()


def generate_questions(doc: dict) -> list:
    prompt = PROMPT_TEMPLATE.format(
        source=doc["source"],
        filename=doc["filename"],
        text=doc["text"],
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content
    return json.loads(raw)["questions"]


# Test on first doc
sample_questions = generate_questions(documents[0])
print("\n".join(sample_questions))

In [ ]:
# Generate for all documents
# Limit to 200 chunks to control cost (~$0.10 at gpt-4o-mini pricing)
SAMPLE_SIZE = 200
sample_docs = documents[:SAMPLE_SIZE]

results = {}

for doc in tqdm(sample_docs):
    doc_id = doc["id"]
    if doc_id in results:
        continue
    try:
        questions = generate_questions(doc)
        results[doc_id] = questions
    except Exception as e:
        print(f"Error for {doc_id}: {e}")
        continue

print(f"Generated questions for {len(results)} chunks")

In [ ]:
# Build ground truth DataFrame
final_results = []
for doc_id, questions in results.items():
    for q in questions:
        final_results.append({"id": doc_id, "question": q})

df_ground_truth = pd.DataFrame(final_results)
df_ground_truth.to_csv("../data/ground-truth-retrieval.csv", index=False)

print(f"Saved {len(df_ground_truth)} question-document pairs")
df_ground_truth.head(10)

## 3. Distribution check

In [ ]:
# Merge with documents to see source distribution
doc_df = pd.DataFrame(documents)
merged = df_ground_truth.merge(doc_df[["id", "source"]], on="id")
merged["source"].value_counts()